# 2.3 · Caching y salida estructurada

**Objetivo:** dos técnicas de producción: abaratar lo que se repite en cada llamada y obtener una salida que otro programa pueda consumir.

**Al terminar tendrás:** el agente del 2.2 con `cache_control` en el system prompt y un resultado final en JSON validado con Pydantic.

Nuestro agente reenvía en cada iteración el mismo system prompt y las mismas herramientas: lo estamos pagando una y otra vez. Y su respuesta final es texto libre: ningún programa puede usarla.

## Preparación

Partimos del 2.2: las dos herramientas y el mapa de despacho.

In [ ]:
%load_ext autoreload
%autoreload 2

from dotenv import load_dotenv
load_dotenv()

import sys; sys.path.append("../..")   # para importar desde la raíz del taller
from pathlib import Path
from config import MODEL

import json
import anthropic
client = anthropic.Anthropic()   # lee ANTHROPIC_API_KEY del entorno

from herramientas import buscar_incidencias, crear_incidencia, restaurar_datos

In [ ]:
TOOLS = [
    {
        "name": "buscar_incidencias",
        "description": (
            "Busca incidencias de soporte en la base de datos. Úsala siempre que el usuario pregunte "
            "por incidencias existentes, su estado o las de un cliente concreto. Devuelve una lista "
            "de incidencias con id, cliente, asunto, estado y prioridad. Si no hay coincidencias devuelve []."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "texto":   {"type": "string", "description": "Texto libre a buscar en asunto y descripción"},
                "estado":  {"type": "string", "enum": ["abierta", "cerrada"]},
                "cliente": {"type": "string", "description": "Nombre exacto del cliente"},
            },
            "required": [],
        },
    },
    {
        "name": "crear_incidencia",
        "description": (
            "Crea una incidencia de soporte nueva y devuelve la incidencia creada con su id. "
            "Úsala solo cuando no exista ya una incidencia abierta sobre el mismo problema "
            "para el mismo cliente: compruébalo antes con buscar_incidencias."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "cliente":     {"type": "string", "description": "Nombre exacto del cliente"},
                "asunto":      {"type": "string", "description": "Resumen del problema en menos de 10 palabras"},
                "descripcion": {"type": "string", "description": "Qué ocurre, desde cuándo y a quién afecta"},
                "prioridad":   {"type": "string", "enum": ["alta", "media", "baja"]},
            },
            "required": ["cliente", "asunto", "descripcion", "prioridad"],
        },
    },
]

FUNCIONES = {"buscar_incidencias": buscar_incidencias, "crear_incidencia": crear_incidencia}

## Un system prompt largo

En un caso real, el system prompt incluye las reglas del negocio. Metemos la política de soporte completa (`datos/politica_soporte.md`):

In [ ]:
INSTRUCCIONES = (
    "Eres un agente de soporte de nivel 1 de una empresa de software. "
    "Respondes en español, de forma breve, y sigues siempre la política de soporte que tienes a continuación."
)
POLITICA = Path("../../datos/politica_soporte.md").read_text(encoding="utf-8")

client.messages.count_tokens(
    model=MODEL, system=INSTRUCCIONES + "\n\n" + POLITICA,
    messages=[{"role": "user", "content": "hola"}],
).input_tokens

Miles de tokens que se envían **en cada iteración** del bucle.

## Prompt caching

Para activar la caché, el system se escribe como una lista de bloques y se marca el último bloque que queremos cachear con `cache_control`:

In [ ]:
SYSTEM = [{
    "type": "text",
    "text": INSTRUCCIONES + "\n\n" + POLITICA,
    "cache_control": {"type": "ephemeral"},
}]

El mismo bucle del 2.2, con una línea más: imprime en cada iteración cuánto se escribe en la caché, cuánto se lee de ella y cuánto se paga a precio normal.

In [ ]:
MAX_ITERACIONES = 10

def ejecutar_agente(pregunta, messages):
    messages.append({"role": "user", "content": pregunta})
    for i in range(MAX_ITERACIONES):
        response = client.messages.create(model=MODEL, max_tokens=4096, system=SYSTEM,
                                          tools=TOOLS, messages=messages)
        u = response.usage
        print(f"  cache_write={u.cache_creation_input_tokens} "
              f"cache_read={u.cache_read_input_tokens} input={u.input_tokens}")
        messages.append({"role": "assistant", "content": response.content})

        if response.stop_reason != "tool_use":
            return next((b.text for b in response.content if b.type == "text"), "")

        resultados = []
        for bloque in response.content:
            if bloque.type != "tool_use":
                continue
            print(f"  [iter {i}] → {bloque.name}({bloque.input})")
            try:
                salida = FUNCIONES[bloque.name](**bloque.input)
                resultados.append({"type": "tool_result", "tool_use_id": bloque.id,
                                   "content": json.dumps(salida, ensure_ascii=False)})
            except Exception as e:
                resultados.append({"type": "tool_result", "tool_use_id": bloque.id,
                                   "content": f"Error: {e}", "is_error": True})
        messages.append({"role": "user", "content": resultados})

    raise RuntimeError("Se alcanzó el máximo de iteraciones")

In [ ]:
historial = []
print(ejecutar_agente("¿Qué incidencias abiertas tiene Initech?", historial))

Primera llamada: `cache_write` alto (se escribe el prefijo en la caché). En las iteraciones siguientes, `cache_read` alto.

Ejecútalo otra vez (o reejecuta la celda de arriba):

In [ ]:
historial = []
print(ejecutar_agente("¿Qué incidencias abiertas tiene Initech?", historial))

Ahora todo el prefijo se **lee** de la caché desde la primera llamada. Leer de la caché cuesta una fracción del precio normal de entrada; escribir en ella, algo más que el precio normal.

### La caché es un prefijo

```
┌─────────┐   ┌──────────┐   ┌────────────┐
│  tools  │ → │  system  │ → │  messages  │
└─────────┘   └──────────┘   └────────────┘
└──────── prefijo cacheado ──┘
```

- La API cachea todo lo que va **desde el principio** hasta el bloque marcado con `cache_control`, en este orden: `tools` → `system` → `messages`.
- Si cambia **un solo byte** del prefijo, la caché no sirve y se vuelve a escribir.
- Hay un **mínimo** de tokens para que un prefijo se cachee: con un system corto no verás nada.

### El anti-patrón: datos variables en el system

Algo muy habitual: poner la fecha y hora actual en el system "para que el modelo sepa qué día es".

In [ ]:
# ❌ FALLO PROVOCADO
from datetime import datetime

SYSTEM_BUENO = SYSTEM
SYSTEM = [{
    "type": "text",
    "text": f"Fecha y hora actual: {datetime.now()}\n\n" + INSTRUCCIONES + "\n\n" + POLITICA,
    "cache_control": {"type": "ephemeral"},
}]
print(ejecutar_agente("¿Qué incidencias abiertas tiene Initech?", []))
SYSTEM = SYSTEM_BUENO

`cache_read=0` en la primera llamada: el prefijo empieza distinto, así que no coincide con nada de lo cacheado. Y cada vez que se ejecute será distinto otra vez. Si hace falta la fecha, va al final, en el mensaje del usuario.

## Salida estructurada

Ahora el agente lo va a invocar un sistema de tickets que necesita un JSON fijo, no texto libre. Definimos el formato con un modelo de Pydantic:

In [ ]:
from pydantic import BaseModel
from typing import Literal

class Resolucion(BaseModel):
    accion: Literal["existente", "creada", "ninguna"]
    incidencia_id: str | None
    resumen_para_cliente: str

Ejecutamos el agente como siempre:

In [ ]:
restaurar_datos()
PREGUNTA = (
    "El cliente Globex dice que no puede exportar a PDF. Si ya hay una incidencia abierta "
    "sobre eso, dime cuál; si no, créala con prioridad media."
)
historial = []
print(ejecutar_agente(PREGUNTA, historial))

Y una última llamada con el historial completo, pidiéndole el resultado en el formato de `Resolucion`. `client.messages.parse` convierte el modelo de Pydantic en un JSON Schema, obliga a la respuesta a cumplirlo y la valida al recibirla.

- Pasamos `tools=TOOLS` porque el historial contiene bloques `tool_use` y `tool_result`: la API exige que las herramientas estén definidas. Además, así el prefijo sigue siendo el mismo y se lee de la caché.
- `tool_choice={"type": "none"}` le indica que ahora no debe pedir herramientas, solo responder.

In [ ]:
final = client.messages.parse(
    model=MODEL, max_tokens=4096, system=SYSTEM, tools=TOOLS,
    tool_choice={"type": "none"},
    messages=historial + [
        {"role": "user", "content": "Resume el resultado en el formato pedido."}
    ],
    output_format=Resolucion,
)
final.parsed_output

`parsed_output` es un objeto de Python validado, no un string que haya que parsear con expresiones regulares:

In [ ]:
resolucion = final.parsed_output
resolucion.accion, resolucion.incidencia_id

In [ ]:
resolucion.model_dump_json(indent=2)

> Si vienes de versiones anteriores: en `messages.create` el parámetro equivalente es `output_config={"format": {"type": "json_schema", "schema": ...}}`. `parse()` lo genera a partir del modelo de Pydantic.

## Resumen del bloque

1. El modelo **pide**; tu código **ejecuta** y devuelve.
2. El bucle es simple pero frágil: ids, orden, errores, límites.
3. Caché y salida estructurada convierten un prototipo en algo usable.